In [1]:
# from selenium import webdriver
# from selenium.webdriver.common.by import By
# from selenium.webdriver.chrome.service import Service
# from selenium.webdriver.common.keys import Keys
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC

# import pandas as pd
# from io import StringIO
# import logging

# # Configure Chrome options
# options = Options()
# # options.add_argument('--headless')  # Run in headless mode (no GUI)
# options.add_argument("--disable-gpu")
# options.add_argument("--no-sandbox")
# options.add_argument("--window-size=1920,1080")

# # Set up the driver
# from seleniumbase import Driver
# from sqlalchemy import create_engine, Column, String, Table, MetaData
# from sqlalchemy.ext.declarative import declarative_base
# from sqlalchemy.orm import sessionmaker
# import json


# SQLITE_FILE_NAME = "./backend/nexa.db"
# DATABASE_URL = f"sqlite:///{SQLITE_FILE_NAME}"

In [ ]:


# Define the ProductInfo class
class ProductInfo:
    def __init__(
        self,
        product_name,
        product_price,
        product_image,
        sizing_table,
        product_properties,
        url,
    ):
        self.product_name = product_name
        self.product_price = product_price
        self.product_image = product_image
        self.url = url
        self.sizing_table = (
            pd.read_html(StringIO(sizing_table), header=0, index_col=0)[0]
            if sizing_table
            else None
        )
        self.product_properties = product_properties

    def __str__(self):
        return f"Product Name: {self.product_name}, Product Price: {self.product_price}, Product Image: {self.product_image}, Sizing Table: {self.sizing_table}"

    def toJson(self):
        sizes = []
        if self.sizing_table is not None:
            for idx, row in self.sizing_table.T.iterrows():
                parameters = []
                for col in self.sizing_table.T.columns:
                    # if col != 'Größen':
                    parameter_value = str(row[col]).strip()
                    if parameter_value.lower() == "nan":
                        parameter_value = ""
                    elif "/" in parameter_value:
                        parts = parameter_value.split("/")
                        if all(part.isdigit() for part in parts):
                            parts = sorted(map(int, parts))
                            if parts == list(range(parts[0], parts[-1] + 1)):
                                parameter_value = str((parts[0] + parts[-1]) / 2)
                                # print(f"changed value: {parameter_value} from {parts}")
                                logging.info(
                                    f"changed value: {parameter_value} from {parts}"
                                )
                            else:
                                # print(f'Warning: {parameter_value} is not a range')
                                logging.warning(
                                    f"Warning: {parameter_value} is not a range"
                                )
                                parameter_value = None
                        parameter_value = str((parts[0] + parts[-1]) / 2)
                    else:
                        try:
                            parameter_value = str(parameter_value)
                        except ValueError:
                            pass
                    parameters.append(
                        {"parameter_name": col, "parameter_value": parameter_value}
                    )
                sizes.append({"size_label": idx, "parameters": parameters})

        return json.dumps(
            {
                "url": self.url,
                "image_url": self.product_image,
                "price": self.product_price,
                "name": self.product_name,
                "properties": self.product_properties,
                "sizes": sizes,
            },
            indent=4,
        )


driver = Driver(uc=True, headless=False)
driver.get("https://hemden.de")
# Accept cookies
accept_cookies_button = driver.find_element(
    By.XPATH, "/html/body/div[2]/div[2]/div[2]/a[2]"
)
accept_cookies_button.click()

In [2]:
i = 1
while True:
    url = f"https://www.hemden.de/herrenhemden-langarm?p={i}&o=3&n=120"
    driver.get(url)
    product_links = driver.find_elements(By.CSS_SELECTOR, 'div.product--box.box--minimal.hmd-list-wrapper a.product--image')
    if len(product_links) == 0:
        break
    product_urls = []
    for link in product_links:
        product_url = link.get_attribute('href')
        product_urls.append(product_url)
    df = pd.DataFrame(product_urls, columns=["Product URL"])
    df.to_csv('langarm_product_urls.csv', mode='a', header=False, index=False)
    i += 1
    if i > 3:
        break

NameError: name 'driver' is not defined

In [5]:
def scrapeProductPage(purl, driver) -> ProductInfo:
        
    # Load the HTML file
    driver.get(purl)


    try:
        # Extract product name
        product_name = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:title"]').get_attribute('content')

        # Extract product price
        product_price = driver.find_element(By.CSS_SELECTOR, 'meta[property="product:price"]').get_attribute('content')

        # Extract product image
        product_image = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:image"]').get_attribute('content')
        # Extract product properties
        WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.product--properties-list-entry')))
        properties_list = driver.find_elements(By.CSS_SELECTOR, '.product--properties-list-entry')
        # print(properties_list)
        product_properties = {}
        for prop in properties_list:
            label = prop.find_element(By.CSS_SELECTOR, '.content--prop-label strong').text.strip(':').strip()
            value = prop.find_element(By.CSS_SELECTOR, '.content--prop-value').text.strip()
            product_properties[label] = value
        # Extract sizing table (if available)
        try:
            table_element = driver.find_element(By.ID, 'product-measures')
            table_html = table_element.get_attribute('outerHTML')  # You can parse this further if needed
        except Exception as e:
            table_html = None  # Handle the case where no table is found

        return ProductInfo(product_name, product_price, product_image, table_html, product_properties, url)
    
    except Exception as e:
        print(f'Error while scraping {url}: {e}')
        return None

In [3]:
import requests
with open('visited_urls.txt', 'r') as f2:
    visited_urls = [url.strip() for url in f2.readlines()]
    # visited_urls = []
with open('langarm_product_urls.csv', 'r') as f:
    urls = f.readlines()
    for url in urls:
        url = url.strip()
        if not url:
            continue
        if url in visited_urls:
            continue
        product_info = scrapeProductPage(url, driver)
        if product_info:
            # print(product_info)
            with open('product.json', 'w') as f3:
                f3.write(product_info.toJson())
                payload = product_info.toJson()
            with open('product.json', 'r') as f4:
                payload = json.load(f4)
            # print(product_info.toJson())
            try:
                response = requests.post("http://localhost:8000/products/", json=payload, headers={"Content-Type": "application/json"})
                if response.status_code == 200:
                    with open('visited_urls.txt', 'a') as f5:
                        f5.write(url + '\n')
                        visited_urls.append(url)
                else:
                    print(f"Error while posting {url}: {response}")
            except Exception as e:
                print(f"Error while posting {url}: {e}")

NameError: name 'scrapeProductPage' is not defined

In [7]:
driver.quit()

In [2]:
from sqlalchemy.orm import Session
from sqlalchemy import select
from sqlalchemy import Table, MetaData
from backend.models import Product, Size, SizeParameter
from sqlalchemy import create_engine


engine = create_engine(DATABASE_URL)
df = pd.DataFrame()
# Create a session
with Session(engine) as session:
    statement = (
        select(Product, Size, SizeParameter)
        .join(Size, Product.id == Size.product_id)
        .join(SizeParameter, Size.id == SizeParameter.size_id)
    )
    results = session.execute(statement).all()
    df = pd.DataFrame(results)
    # for product, size, size_parameter in results:
        # print(f"Product: {product.name}, Size: {size.size_label}, Parameter: {size_parameter.parameter_name} = {size_parameter.parameter_value}")

In [1]:
engine = create_engine(DATABASE_URL)
products = pd.read_sql_table('product', engine)
# df = pd.read_sql_query('SELECT * FROM product JOIN size ON product.id = size.product_id JOIN SizeParameter ON size.id = SizeParameter.size_id', engine)
sizes = pd.read_sql_table('size', engine)
size_parameters = pd.read_sql_table('sizeparameter', engine)

NameError: name 'create_engine' is not defined

In [16]:
products.to_csv('dump/products.csv')
sizes.to_csv('dump/sizes.csv')
size_parameters.to_csv('dump/size_parameters.csv')

In [5]:
sizes['size_label'].value_counts()

size_label
M              247
L              247
XL             247
XXL            247
3XL            247
4XL            247
5XL            243
6XL            243
S              236
XS             232
Unnamed: 9      11
Unnamed: 10     11
7XL              9
Name: count, dtype: int64

In [6]:
sizes.loc[sizes['size_label'] == 'Unnamed: 9']

,id,product_id,size_label
148,149,17,Unnamed: 9
179,180,20,Unnamed: 9
279,280,31,Unnamed: 9
309,310,34,Unnamed: 9
379,380,42,Unnamed: 9
449,450,51,Unnamed: 9
489,490,56,Unnamed: 9
639,640,73,Unnamed: 9
771,772,88,Unnamed: 9
962,963,114,Unnamed: 9


In [27]:
products.head()
sizes.pivot(index='product_id', columns='size_label', values='id')

size_label,3XL,4XL,5XL,6XL,7XL,L,M,S,Unnamed: 10,Unnamed: 9,XL,XS,XXL
product_id,,,,,,,,,,,,,
1,7.0,8.0,9.0,10.0,NaN,4.0,3.0,2.0,NaN,NaN,5.0,1.0,6.0
2,17.0,18.0,19.0,20.0,NaN,14.0,13.0,12.0,NaN,NaN,15.0,11.0,16.0
3,27.0,28.0,29.0,30.0,NaN,24.0,23.0,22.0,NaN,NaN,25.0,21.0,26.0
4,37.0,38.0,39.0,40.0,NaN,34.0,33.0,32.0,NaN,NaN,35.0,31.0,36.0
5,47.0,48.0,49.0,50.0,NaN,44.0,43.0,42.0,NaN,NaN,45.0,41.0,46.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
287,2424.0,2425.0,2426.0,2427.0,NaN,2421.0,2420.0,2419.0,NaN,NaN,2422.0,2418.0,2423.0
288,2434.0,2435.0,2436.0,2437.0,NaN,2431.0,2430.0,2429.0,NaN,NaN,2432.0,2428.0,2433.0
289,2444.0,2445.0,2446.0,2447.0,NaN,2441.0,2440.0,2439.0,NaN,NaN,2442.0,2438.0,2443.0


In [39]:
size_parameters.parameter_name.value_counts()

parameter_name
Kragenweite                2439
Oberweite                  2439
Taillenweite               2439
Hemdenlänge:               2399
bei Armlänge normal        2399
bei Armlänge extra lang    1839
Rückenbreite               1140
bei Armlänge extra kurz    1100
Schulterbreite              519
Hüftweite                   390
Kragenweite, cm              28
Oberweite, cm                28
Taillenweite, cm             28
Rückenlänge                  28
Name: count, dtype: int64

In [13]:
size_parameters[size_parameters['parameter_name'].str.contains('Armlänge|Oberweite', case=False, na=False)].groupby('size_id').apply(lambda x: x.to_dict(orient='records')).to_dict()[1]

/var/folders/lb/40wh6kb178s6fcvv5ds7hxg80000gn/T/ipykernel_8577/4028574225.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  size_parameters[size_parameters['parameter_name'].str.contains('Armlänge|Oberweite', case=False, na=False)].groupby('size_id').apply(lambda x: x.to_dict(orient='records')).to_dict()[1]


[{'id': 2,
  'size_id': 1,
  'parameter_name': 'Oberweite',
  'parameter_value': 98.0},
 {'id': 6,
  'size_id': 1,
  'parameter_name': 'bei Armlänge extra kurz',
  'parameter_value': 76.0},
 {'id': 7,
  'size_id': 1,
  'parameter_name': 'bei Armlänge normal',
  'parameter_value': 76.0},
 {'id': 8,
  'size_id': 1,
  'parameter_name': 'bei Armlänge extra lang',
  'parameter_value': 79.0}]

In [ ]:
sizes = size_parameters.groupby('size_id').apply(lambda x: x.to_dict(orient='records')).to_dict()

# Normalize the dictionary and create a DataFrame
normalized_sizes = []
for size_id, params in sizes.items():
	for param in params:
		param['size_id'] = size_id
		normalized_sizes.append(param)

sizesdf = pd.DataFrame(normalized_sizes)
sizesdf

/var/folders/lb/40wh6kb178s6fcvv5ds7hxg80000gn/T/ipykernel_8577/3726910602.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sizes = size_parameters.groupby('size_id').apply(lambda x: x.to_dict(orient='records')).to_dict()


,id,size_id,parameter_name,parameter_value
0,1,1,Kragenweite,36.0
1,2,1,Oberweite,98.0
2,3,1,Taillenweite,86.0
3,4,1,Rückenbreite,38.0
4,5,1,Hemdenlänge:,NaN
...,...,...,...,...
17210,17211,2467,Oberweite,NaN
17211,17212,2467,Taillenweite,NaN
17212,17213,2467,Hemdenlänge:,NaN
17213,17214,2467,bei Armlänge normal,NaN
